In [2]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn.functional as F
from MedSAM.segment_anything import sam_model_registry
from MedSAM.MedSAM_Inference import medsam_inference
from skimage import io, transform
from tqdm import tqdm

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
medsam_model = sam_model_registry["vit_b"](checkpoint="MedSAM/work_dir/MedSAM/medsam_vit_b.pth")
medsam_model = medsam_model.to(device)
medsam_model.eval()

/home/paulb/INF8225/Projet/MedSAM/segment_anything/build_sam.py:144: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(f, map_location=torch.device('cpu'

Sam(
  (image_encoder): ImageEncoderViT(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (blocks): ModuleList(
      (0-11): 12 x Block(
        (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=768, out_features=2304, bias=True)
          (proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (mlp): MLPBlock(
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): Linear(in_features=3072, out_features=768, bias=True)
          (act): GELU(approximate='none')
        )
      )
    )
    (neck): Sequential(
      (0): Conv2d(768, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (1): LayerNorm2d()
      (2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (3): LayerNorm2d()
    )


In [4]:
def calculate_dice(mask_true, mask_pred):
    m_true = np.asarray(mask_true).astype(bool)
    m_pred = np.asarray(mask_pred).astype(bool)

    if m_true.sum() + m_pred.sum() == 0:
        return 1.0
    
    intersection = np.logical_and(m_true, m_pred).sum()
    return 2 * intersection / (m_true.sum() + m_pred.sum())

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
from skimage import io, transform
from tqdm import tqdm

# --- Configuration des chemins MSD ---
base_dir = "data/MSD_pancreas"
output_folder = os.path.join(base_dir, "outputs_baseline")
os.makedirs(output_folder, exist_ok=True)
os.makedirs("data/results", exist_ok=True)

# Chargement du fichier maître COCO
with open(os.path.join(base_dir, "annotations.json")) as f:
    coco_data = json.load(f)

# Groupement des annotations par image_id
annotations_by_image = {}
for ann in coco_data["annotations"]:
    img_id = ann["image_id"]
    if img_id not in annotations_by_image:
        annotations_by_image[img_id] = []
    annotations_by_image[img_id].append(ann)

dice_list = []

for img_info in tqdm(coco_data["images"], desc="Baseline MedSAM (Grayscale Z)"):
    img_id = img_info["id"]
    img_rel_path = img_info["file_name"] 
    img_path = os.path.join(base_dir, img_rel_path)
    
    mask_rel_path = img_rel_path.replace("/images/", "/masks/")
    mask_path = os.path.join(base_dir, mask_rel_path)

    # Déterminer si l'image contient une tumeur (Catégorie 2)
    has_tumor = False
    if img_id in annotations_by_image:
        has_tumor = any(ann["category_id"] == 2 for ann in annotations_by_image[img_id])

    # Lecture de l'image (skimage lit en RGB, donc [Global Z, Focal Z, Focal Z+1])
    img_np = io.imread(img_path)
    
    # Transformation en Grayscale Repeat sur la coupe Z
    # Index 0 = Global Z (W400), Index 1 = Focal Z (W150), Index 2 = Focal Z+1 (W150)
    # On sélectionne l'index 1 pour donner à MedSAM le meilleur contraste possible
    img_z_gray = img_np[:, :, 1]
    
    # Duplication sur 3 canaux identiques
    img_3c = np.stack([img_z_gray, img_z_gray, img_z_gray], axis=-1)
    
    H, W, _ = img_3c.shape

    # Préparation MedSAM
    img_1024 = transform.resize(
        img_3c, (1024, 1024), order=3, preserve_range=True, anti_aliasing=True
    ).astype(np.uint8)
    
    img_1024 = (img_1024 - img_1024.min()) / np.clip(
        img_1024.max() - img_1024.min(), a_min=1e-8, a_max=None
    ) 
    img_1024_tensor = (
        torch.tensor(img_1024).float().permute(2, 0, 1).unsqueeze(0).to(device)
    )

    with torch.no_grad():
        image_embedding = medsam_model.image_encoder(img_1024_tensor)

    full_medsam_seg = np.zeros((H, W), dtype=np.uint8)

    # Si l'image a une tumeur, on itère sur ses boîtes pour guider MedSAM
    if has_tumor:
        for ann in annotations_by_image[img_id]:
            if ann["category_id"] != 2: 
                continue
                
            x, y, w, h = ann["bbox"]
            box_np = np.array([[x, y, x + w, y + h]]) 
            box_1024 = box_np / np.array([W, H, W, H]) * 1024

            medsam_seg = medsam_inference(medsam_model, image_embedding, box_1024, H, W)
            full_medsam_seg[medsam_seg > 0] = 1

    # Sauvegarde du masque prédit
    io.imsave(
        os.path.join(output_folder, "seg_" + os.path.basename(img_path)),
        (full_medsam_seg * 255).astype(np.uint8),
        check_contrast=False,
    )

    # Chargement du masque vérité terrain
    true_seg_raw = io.imread(mask_path)
    true_seg = np.zeros_like(true_seg_raw, dtype=np.uint8)
    true_seg[true_seg_raw == 2] = 1

    # Calcul du DICE
    if not has_tumor:
        dice_score = 1.0
    else:
        dice_score = calculate_dice(true_seg, full_medsam_seg)

    dice_list.append({
        "image_name": os.path.basename(img_path),
        "split": img_info["split"],
        "has_tumor": has_tumor,
        "dice": dice_score
    })
    
df = pd.DataFrame(dice_list)
df.to_csv("data/results/dice_baseline_msd.csv", index=False)

Baseline MedSAM (Grayscale Z): 100%|██████████| 1000/1000 [17:13<00:00,  1.03s/it]


In [5]:
import os
import json
import numpy as np
import pandas as pd
import torch
from skimage import io, transform
from tqdm import tqdm

# (Assure-toi d'avoir importé et initialisé medsam_model et device avant)

# --- Configuration des chemins MSD ---
base_dir = "data/MSD_pancreas"
output_folder = os.path.join(base_dir, "outputs_baseline_2_5D") # Nouveau dossier
os.makedirs(output_folder, exist_ok=True)
os.makedirs("data/results", exist_ok=True)

# Chargement du fichier maître COCO
with open(os.path.join(base_dir, "annotations.json")) as f:
    coco_data = json.load(f)

# Groupement des annotations par image_id
annotations_by_image = {}
for ann in coco_data["annotations"]:
    img_id = ann["image_id"]
    if img_id not in annotations_by_image:
        annotations_by_image[img_id] = []
    annotations_by_image[img_id].append(ann)

dice_list = []

for img_info in tqdm(coco_data["images"], desc="Baseline Oracle MedSAM (2.5D)"):
    img_id = img_info["id"]
    img_rel_path = img_info["file_name"] 
    img_path = os.path.join(base_dir, img_rel_path)
    
    mask_rel_path = img_rel_path.replace("/images/", "/masks/")
    mask_path = os.path.join(base_dir, mask_rel_path)

    has_tumor = False
    if img_id in annotations_by_image:
        has_tumor = any(ann["category_id"] == 2 for ann in annotations_by_image[img_id])

    # =========================================================
    # MODIFICATION 2.5D : EXPLOITATION DES 3 CANAUX HYBRIDES
    # =========================================================
    # Lecture de l'image encodée lors de l'extraction
    # Index 0 (R) = Global Z, Index 1 (G) = Focal Z, Index 2 (B) = Focal Z+1
    img_np = io.imread(img_path)
    
    # On garde l'image telle quelle avec ses 3 canaux d'information distincts
    if len(img_np.shape) == 2:
        img_3c = np.repeat(img_np[:, :, None], 3, axis=-1) # Sécurité au cas où
    else:
        img_3c = img_np
    # =========================================================

    H, W, _ = img_3c.shape

    # Préparation MedSAM
    img_1024 = transform.resize(
        img_3c, (1024, 1024), order=3, preserve_range=True, anti_aliasing=True
    ).astype(np.uint8)
    
    img_1024 = (img_1024 - img_1024.min()) / np.clip(
        img_1024.max() - img_1024.min(), a_min=1e-8, a_max=None
    ) 
    img_1024_tensor = (
        torch.tensor(img_1024).float().permute(2, 0, 1).unsqueeze(0).to(device)
    )

    with torch.no_grad():
        image_embedding = medsam_model.image_encoder(img_1024_tensor)

    full_medsam_seg = np.zeros((H, W), dtype=np.uint8)

    if has_tumor:
        for ann in annotations_by_image[img_id]:
            if ann["category_id"] != 2: 
                continue
                
            x, y, w, h = ann["bbox"]
            box_np = np.array([[x, y, x + w, y + h]]) 
            box_1024 = box_np / np.array([W, H, W, H]) * 1024

            medsam_seg = medsam_inference(medsam_model, image_embedding, box_1024, H, W)
            full_medsam_seg[medsam_seg > 0] = 1

    io.imsave(
        os.path.join(output_folder, "seg_" + os.path.basename(img_path)),
        (full_medsam_seg * 255).astype(np.uint8),
        check_contrast=False,
    )

    true_seg_raw = io.imread(mask_path)
    true_seg = np.zeros_like(true_seg_raw, dtype=np.uint8)
    true_seg[true_seg_raw == 2] = 1

    if not has_tumor:
        dice_score = 1.0
    else:
        # Assure-toi que calculate_dice est définie dans ton script
        dice_score = calculate_dice(true_seg, full_medsam_seg)

    dice_list.append({
        "image_name": os.path.basename(img_path),
        "split": img_info["split"],
        "has_tumor": has_tumor,
        "dice": dice_score
    })
    
df = pd.DataFrame(dice_list)
df.to_csv("data/results/dice_baseline_msd_25D.csv", index=False)

Baseline Oracle MedSAM (2.5D): 100%|██████████| 1000/1000 [16:55<00:00,  1.02s/it]


In [6]:
df = pd.read_csv("data/results/dice_baseline_msd_25D.csv")
df.head()

,image_name,split,has_tumor,dice
0,pancreas_001_s40.png,train,True,0.929362
1,pancreas_001_s41.png,train,True,0.769982
2,pancreas_001_s44.png,train,True,0.851863
3,pancreas_001_s57.png,train,False,1.000000
4,pancreas_001_s60.png,train,False,1.000000


In [7]:
# Chargement du fichier CSV
df = pd.read_csv("data/results/dice_baseline_msd_25D.csv")

# 1. Statistiques Globales (Toutes les images)
print("=== STATISTIQUES GLOBALES ===")
print(df['dice'].describe())
print("\n")

# 2. Statistiques sur les images AVEC tumeur uniquement (Le vrai score de détection)
df_tumor = df[df['has_tumor'] == True]
print("=== STATISTIQUES (AVEC TUMEUR UNIQUEMENT) ===")
print(df_tumor['dice'].describe())
print("\n")

# 3. Statistiques par Split (Optionnel, très utile pour le rapport)
print("=== DSC MOYEN PAR SPLIT (Tumeurs Uniquement) ===")
print(df_tumor.groupby('split')['dice'].mean())

=== STATISTIQUES GLOBALES ===
count    1000.000000
mean        0.875278
std         0.167365
min         0.180698
25%         0.787171
50%         0.987844
75%         1.000000
max         1.000000
Name: dice, dtype: float64


=== STATISTIQUES (AVEC TUMEUR UNIQUEMENT) ===
count    500.000000
mean       0.750557
std        0.157812
min        0.180698
25%        0.662683
50%        0.786865
75%        0.875438
max        0.975687
Name: dice, dtype: float64


=== DSC MOYEN PAR SPLIT (Tumeurs Uniquement) ===
split
test     0.754985
train    0.746997
val      0.774605
Name: dice, dtype: float64
